
# Knowledge Graphs and Semantic Technologies -- ML4KG Tutorial


Import libraries

In [1]:
# you need to install pykeen beforehand, see https://pykeen.readthedocs.io/en/stable/installation.html 
import os
import numpy as np
import pandas as pd
import pykeen
import seaborn
import zipfile
import io
from pykeen import datasets
from pykeen.pipeline import pipeline
from rdflib import Graph, ConjunctiveGraph, Literal, BNode, Namespace, RDF, URIRef, RDFS
import urllib.parse
import sys
import pathlib
from urllib.parse import urlparse

sys.modules["pathlib._local"] = pathlib
if os.name == 'nt':
   pathlib.PosixPath = pathlib.WindowsPath
else:
   pathlib.WindowsPath = pathlib.PosixPath



/opt/anaconda3/envs/knowledge_graphs/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Dataset exploration


PyKeen comes with its own datasets that can be used directly in a pipeline.
Below we import it so that we can explore it later.

In [2]:
from pykeen.datasets import Nations

However, we want to be able tfo work with our own datasets as well, so we etch the online French Royalty dataset.

In [3]:
import requests
from pykeen import triples

url = 'https://github.com/halliwelln/multiple-explanations/raw/refs/heads/main/data/french_royalty.npz.zip'
response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall("data/")

royals = np.load('data/french_royalty.npz')['all_triples']

# PyKeen uses the TriplesFactory class to store the triples. It stores triples by mapping entities and relations into integer identifiers.
royals= triples.TriplesFactory.from_labeled_triples(royals)
royals_triples = royals.triples # PyKeen won't like it because it's costly and usually not needed if LP algorithms are to be trained on these.
royals_triples[:5,]

Reconstructing all label-based triples. This is expensive and rarely needed.


array([['<http://dbpedia.org/resource/Adalard_of_Paris>', 'child',
        '<http://dbpedia.org/resource/Adelaide_of_Paris>'],
       ['<http://dbpedia.org/resource/Adalbert_I_of_Ivrea>', 'child',
        '<http://dbpedia.org/resource/Berengar_II_of_Italy>'],
       ['<http://dbpedia.org/resource/Adalbert_of_Italy>', 'child',
        '<http://dbpedia.org/resource/Otto-William,_Count_of_Burgundy>'],
       ['<http://dbpedia.org/resource/Adalbert_of_Italy>', 'grandparent',
        '<http://dbpedia.org/resource/Adalbert_I_of_Ivrea>'],
       ['<http://dbpedia.org/resource/Adalbert_of_Italy>', 'grandparent',
        '<http://dbpedia.org/resource/Boso,_Margrave_of_Tuscany>']],
      dtype='<U92')

### Exercise 1

List the unique subject and object entities found in the dataset. Then list all of the relationships that link the entities (note that some entities are not linked). Create an RDF version of the dataset, using your own namespaces, and save is as a ttl file. 

Using SPARQL, answer the following questions : 
1. How many instances per class? Use ORDER BY to show the most popular class
2. What is the most common relation per each class?

In [6]:
subjects = sorted(set(royals_triples[:, 0]))
objects  = sorted(set(royals_triples[:, 2]))
entities = sorted(set(subjects) | set(objects))
relations = sorted(set(royals_triples[:, 1]))

print("Unique subjects:", len(subjects))
print("Unique objects :", len(objects))
print("Unique entities:", len(entities))
print("Unique relations:", len(relations))

print("\nFirst 10 entities:\n", entities[:10])
print("\nRelations:\n", relations)

Unique subjects: 2125
Unique objects : 2120
Unique entities: 2125
Unique relations: 6

First 10 entities:
 [np.str_('<http://dbpedia.org/resource/Adalard_of_Paris>'), np.str_('<http://dbpedia.org/resource/Adalbert_I_of_Ivrea>'), np.str_('<http://dbpedia.org/resource/Adalbert_of_Italy>'), np.str_('<http://dbpedia.org/resource/Adam_Albert_von_Neipperg>'), np.str_('<http://dbpedia.org/resource/Adam_FitzRoy>'), np.str_('<http://dbpedia.org/resource/Adam_Gordon_of_Auchindoun>'), np.str_('<http://dbpedia.org/resource/Adela_of_Champagne>'), np.str_('<http://dbpedia.org/resource/Adela_of_France>'), np.str_('<http://dbpedia.org/resource/Adela_of_Milan>'), np.str_('<http://dbpedia.org/resource/Adela_of_Normandy>')]

Relations:
 [np.str_('brother'), np.str_('child'), np.str_('grandparent'), np.str_('parent'), np.str_('sister'), np.str_('spouse')]


In [9]:
def fixURI(entity):
    text = entity.strip('>').strip('<')
    parts= text.split('/')
    parts[-1] = urllib.parse.quote(parts[-1])
    return URIRef('/'.join(parts))

g = Graph()
REL = Namespace("http://example.org/royals/relation/")
g.bind("rel", REL)

for s, p, o in royals_triples:
    g.add((fixURI(s), REL[p], fixURI(o)))

g.serialize("data/royalty.ttl", format="turtle")

<Graph identifier=Nb655948ca1004493bdce24659771d830 (<class 'rdflib.graph.Graph'>)>

In [14]:
r_graph = Graph()
r_graph.parse('data/royalty.ttl')

<Graph identifier=N7510b036117b4abab491ab9f70994c95 (<class 'rdflib.graph.Graph'>)>

In [ ]:
def Query1(graph):
   res = graph.query("""
    select ?subj (count(?subj) AS ?count) where { 
    ?subj ?pred ?obj
        } group by ?subj order by desc(?count)
   """,
   initNs={'roy':'http://royalties/','db':'http://dbpedia.org/resource/'})

   return list(res)

print('The most popular classes:')
Query1(r_graph)

The most popular classes:


[(rdflib.term.URIRef('http://dbpedia.org/resource/Ferdinand_I%2C_Holy_Roman_Emperor'),
  rdflib.term.Literal('30', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://dbpedia.org/resource/Louis_VII_of_France'),
  rdflib.term.Literal('30', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://dbpedia.org/resource/Charles%2C_Count_of_Valois'),
  rdflib.term.Literal('26', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://dbpedia.org/resource/Eleanor_of_Austria'),
  rdflib.term.Literal('26', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://dbpedia.org/resource/Maria_Carolina_of_Austria'),
  rdflib.term.Literal('25', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://dbpedia.org/resource/Eleanor_of_Aquitaine'),
  rdflib.term.Litera

In [20]:
def Query2(graph):
    res = graph.query("""
    select distinct ?pred (count(?pred) AS ?count) where { 
    ?s ?pred ?obj
        } group by ?pred order by desc(?count)
    """,initNs={'roy':'http://royalties/','db':'http://dbpedia.org/resource/'})
    return list(res)  

print('The most common relationship per class:')
Query2(r_graph)  

The most common relationship per class:


[(rdflib.term.URIRef('http://example.org/royals/relation/grandparent'),
  rdflib.term.Literal('5095', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://example.org/royals/relation/child'),
  rdflib.term.Literal('3137', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://example.org/royals/relation/parent'),
  rdflib.term.Literal('2637', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://example.org/royals/relation/spouse'),
  rdflib.term.Literal('1172', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://example.org/royals/relation/brother'),
  rdflib.term.Literal('179', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://example.org/royals/relation/sister'),
  rdflib.term.Literal('137', datatype=rdflib.term.URIRef('http://www.

# 2. Defining train and test datasets


As is typical in machine learning, we need to split our dataset into training and test (and sometimes validation) datasets.

What differs from the standard method of randomly sampling N points to make up our test set, is that our data points are two entities linked by some relationship, and we need to take care to ensure that all entities are represented in train and test sets by at least one triple.

To accomplish this, PyKEEN provides the <b>pykeen.triples.TriplesFactory.split()</b> function, which defaults to an 80/20 split. It is also by default stratified, to ensure that the distribution of the test set corresponds to that of the training set. If you want to use early stopping, you will also need a validation set. The function takes a list of percentages as argument: if you want a 95/5 split you give it <b>[0.95,0.05]</b> as argument, if you want 90/5/5 (which would include a validation set as well) you give it <b>[0.9,0.05,0.05]</b> as argument and it will return 3 datasets.

For sake of example, we will create a small test size that includes only 5% of triples. 

In [21]:
royals_training, royals_testing = royals.split([0.95,0.05], random_state=42) #random state can be used for reproducitbility

print('Train set size: ', royals_training.mapped_triples.shape)
print('Test set size: ', royals_testing.mapped_triples.shape)

Train set size:  torch.Size([11739, 3])
Test set size:  torch.Size([618, 3])


### Exercise 2

Create three train-test sets of different sizes from the GoT data. Give them different names. Make sure the test set is not too big when compared to the training set (test set should be max 15% of the total dataset).

In [22]:
royals_training_80, royals_testing_80 = royals.split([0.8,0.2], random_state=42)

royals_training_90, royals_testing_90 = royals.split([0.9,0.1], random_state=42)

royals_training_85, royals_testing_85 = royals.split([0.85,0.15], random_state=42)

# 3. Training and testing the model

PyKEEN has implemented several Knoweldge Graph Embedding models (TransE, ComplEx, DistMult, HolE, etc.). We will use the ComplEx model with default values for this tutorial.

You can find the list of all implemented models in the documentation: https://pykeen.readthedocs.io/en/stable/reference/models.html

Importing a model and instantiate it:
There are two ways to import and use a model, both are shown below and don't give different results but not importing the model before hand might cause the automatic importing to be slower, especially if you plan to use the same model multiple times.

In [23]:
# we need the pipeline to run a model, so it is simpler to import it directly.
# Pykeen lets you train a model with the minimal amount of custom parameters

from pykeen.pipeline import pipeline

# here we don't import the model, but let PyKEEN do the importing.
pipeline_result_simple = pipeline(
    random_seed=0,
    model='ComplEx',
    training=royals_training,
    testing=royals_testing,
)
pipeline_result_simple.plot_losses()

No cuda devices were available. The model runs on CPU
/opt/anaconda3/envs/knowledge_graphs/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Training epochs on cpu:   0%|          | 0/5 [00:00<?, ?epoch/s]

: 

In [ ]:
# here we import the model and use it directly.
from pykeen.models import ComplEx

pipeline_result_imported = pipeline(
    random_seed=0,
    model=ComplEx,
    training=royals_training,
    testing=royals_testing,
)
pipeline_result_imported.plot_losses()

You can retrieve different metrics from the results. Here we retrieve the mean reciprocal rank (MRR). The result is the same for both the simple and imported model, because we used the same random seed (0).

In [ ]:
print(pipeline_result_imported.get_metric('mrr'))
print(pipeline_result_simple.get_metric('mrr'))

In [ ]:
# but to get a better performing model, you want to set different things
pipeline_result = pipeline(
    random_seed=0,
    model='ComplEx',
    training=royals_training,
    testing=royals_testing,
    epochs=200,
    dimensions=150,
    optimizer='adam',
    optimizer_kwargs={'lr':1e-3},
    loss='pairwisehinge', 
    regularizer='LP', 
    regularizer_kwargs={'p':3, 'weight':1e-5}
)
print(pipeline_result.get_metric('mrr'))

Understanding the parameters:

- dimensions : the dimensionality of the embedding space
- negative_sampler : the negative samplic strategy, here set to default (not used in arguments).
- batch_size : the number of batches in which the training set is split during the training loop. If you are having into low memory issues than settings this to a higher number may help.
- epochs : the number of epochs to train the model for.
- optimizer : the Adam optimizer, with a learning rate of $1e-3$ set via the <i>optimizer_kwarg</i>.
- loss : pairwise loss, with a margin of $0.5$ set via the <i>loss_kwarg</i>.
- regularizer :  regularization with $p=2$, i.e. $l_2$ regularization. $\lambda$ = $1e-5$, set via the <i>regularizer_kwarg</i>.

### Filtering Negatives

To ensure our model can be trained and evaluated correctly, we need to define a filter to ensure that no negative statements generated by the corruption procedure are actually positives. This is simply done by concatenating train and test sets. When negative triples are generated by the corruption strategy, we can check that they aren't actually true statements.

With PyKEEN this is made very easy, and can simply be passed as an argument.

In [ ]:
pipeline_result = pipeline(
    model='ComplEx',
    training=royals_training,
    testing=royals_testing,
    epochs=200,
    dimensions=150,
    optimizer='adam',
    optimizer_kwargs={'lr':1e-3},
    loss='pairwisehinge', 
    regularizer='LP', 
    regularizer_kwargs={'p':3, 'weight':1e-5}, 
    
    negative_sampler='basic',
    negative_sampler_kwargs=dict(
        filtered=True,
    )
)
print(pipeline_result.get_metric('mrr'))

To save your learned model and also the results, we need to add checkpoints to the pipeline.
By adding training kwargs to the pipeline, the model will be automatically saved. By default, it saves the model after every epoch (checkpoint_frequency=0). You can also set the directory to which the models are saved, but by default they will end up in ~/.data/pykeen/checkpoints.

In [ ]:
pipeline_result = pipeline(
    model='ComplEx',
    training=royals_training,
    testing=royals_testing,
    training_kwargs=dict(
        num_epochs=200,
        checkpoint_name='royals_complex_checkpoint.pt',
        checkpoint_directory='checkpoint_dir/',
        checkpoint_frequency=20,
    ),
    dimensions=150,
    optimizer='adam',
    optimizer_kwargs={'lr':1e-3},
    loss='pairwisehinge', 
    regularizer='LP', 
    regularizer_kwargs={'p':3, 'weight':1e-5}, 
    negative_sampler='basic',
    negative_sampler_kwargs=dict(
        filtered=True,
    )
)

There is another way to save models, but for that we need to do the training and evaluating outside of the pipeline model. Below is an example of the above model training outside of the pipeline module.

In [ ]:
from pykeen.models import ComplEx
model = ComplEx(triples_factory=royals_training)

from pykeen.optimizers import Adam
optimizer = Adam(params=model.get_grad_params())

# from pykeen.regularizers import LP
# regularizer = LP(p=3,weight=1e-5)

from pykeen.training import SLCWATrainingLoop
training_loop = SLCWATrainingLoop(model=model,
                                  triples_factory=royals_training,
                                  optimizer=optimizer)

#training
_ = training_loop.train(triples_factory=royals_training,
                    num_epochs=200)

#evaluating
from pykeen.evaluation import RankBasedEvaluator
evaluator = RankBasedEvaluator()
mapped_triples = royals_testing.mapped_triples

results = evaluator.evaluate(
            model=model,
            mapped_triples=mapped_triples,
            )

print(results.get_metric('mrr'))

#save results, this works also with the pipeline results, as the results object 
#returned by the evaluator is the same as the one returned from the pipeline
save_dir = 'royals_complex'
if not os.path.isdir(save_dir):
    os.mkdir(save_dir)
results.to_df().to_csv(save_dir+os.path.sep+'results.csv')

import torch
torch.save(model,'trained_model.pkl')

#to load the model use the following command
# my_pykeen_model = torch.load('trained_model.pkl')

### Exercise 3

Try changing the parameters of your training process. See if you obtain a better model in terms of average loss. Save it as ./data/best_model.pkl. Which parameters work best for the dataset?

Now use the training and test set you created in Exercise 2. Which loss you obtain, and for which parameters?

Remember to save each model locally with a different name, so you can find them back.

In [ ]:
### your code here

# 4. Evaluating the Model

### Metrics

We can now get some evaluation metrics for our model, they were already computed during evaluation time as part of the pipeline, and print them out.

We are going to use the following evaluation metrics:
- <i>mrr</i> (mean reciprocal rank) : this function computes the mean of the reciprocal of elements of a vector of rankings ranks
- <i>hits_at_n</i> : this function computes how many elements of a vector of rankings ranks make it to the $top_n$ positions.

NB : The choice of which _N_ makes more sense depends on the application and the size of the dataset.

In [ ]:
pipeline_result.get_metric('hits_at_10')

In [ ]:
# from ampligraph.evaluation import mr_score, mrr_score, hits_at_n_score

mrr = pipeline_result.get_metric('mrr')
print("MRR: %.4f" % (mrr))
print()

hits_10 = pipeline_result.get_metric('hits_at_10')
print("Hits@10: %.6f" % (hits_10))
print("Interpretation: on average, the model guessed the correct subject or object %.1f%% of the time when considering the top-10 better ranked triples.\n" % (hits_10*100))

hits_3 = pipeline_result.get_metric('hits_at_3')
print("Hits@3: %.6f" % (hits_3))
print("Interpretation: on average, the model guessed the correct subject or object %.1f%% of the time when considering the top-3 better ranked triples.\n" % (hits_3*100))

# hits_1 = hits_at_n_score(ranks, n=1)
# print("Hits@1: %.2f" % (hits_1))
# print("Interpretation: on average, the model guessed the correct subject or object %.1f%% of the time when considering the top-1 better ranked triples.\n" % (hits_1*100))


### Exercise 4

Evaluate the models you created before (different set sizes, different parameters). Summarise your results in a table.

In [ ]:
### your code here

# 5. Link Prediction

Link prediction allows to infer missing links in a graph. This has many real-world use cases, such as predicting connections between people in a social network, interactions between proteins in a biological network, and music recommendation based on prior user taste.

In our case, we are going to see which of the following candidate statements is more likely to be true. Note that the candidate statements below are made up, i.e. they are not in the original dataset.

We will also use a simpler dataset based on the (unfortunate) TV series Game of Thrones.

In [ ]:
# this is the original link, nowadays it is denying access
# url = 'https://ampligraph.s3-eu-west-1.amazonaws.com/datasets/GoT.csv'
# Format that can be read by a pd.from_csv should also be able to be read here, but the delimiter needs to be adjusted

got = triples.TriplesFactory.from_path('../data/GoT.csv',load_triples_kwargs=dict(delimiter=','))# PyKEEN uses tabs as defaults

In [ ]:
from pykeen.datasets import Nations

pipeline_result = pipeline(
    model='complex',
    training=got,
    testing=got, #a bit of a cheat, but here we are just predicting for fun and we will use the full graph
    training_kwargs=dict(
        num_epochs=100),
    dimensions=150,
    optimizer='adam',
    optimizer_kwargs={'lr':1e-3},
    loss='pairwisehinge',
    #regularizer='LP',
    regularizer_kwargs={'p':3, 'weight':1e-5},
    negative_sampler='basic',
    negative_sampler_kwargs=dict(
        filtered=True,
    )
)


In [ ]:
X_unseen = np.array([
    ['Jorah Mormont', 'SPOUSE', 'Daenerys Targaryen'],
    ['Tyrion Lannister', 'SPOUSE', 'Missandei'],
    ["King's Landing", 'SEAT_OF', 'House Lannister of Casterly Rock'],
    ['Sansa Stark', 'SPOUSE', 'Petyr Baelish'],
    ['Daenerys Targaryen', 'SPOUSE', 'Jon Snow'],
    ['Daenerys Targaryen', 'SPOUSE', 'Craster'],
    ['House Stark of Winterfell', 'IN_REGION', 'The North'],
    ['House Stark of Winterfell', 'IN_REGION', 'Dorne'],
    ['House Tyrell of Highgarden', 'IN_REGION', 'Beyond the Wall'],
    ['Brandon Stark', 'ALLIED_WITH', 'House Stark of Winterfell'],
    ['Brandon Stark', 'ALLIED_WITH', 'House Lannister of Casterly Rock'],
    ['Rhaegar Targaryen', 'PARENT_OF', 'Jon Snow'],
    ['House Hutcheson', 'SWORN_TO', 'House Tyrell of Highgarden'],
    ['Daenerys Targaryen', 'ALLIED_WITH', 'House Stark of Winterfell'],
    ['Daenerys Targaryen', 'ALLIED_WITH', 'House Lannister of Casterly Rock'],
    ['Jaime Lannister', 'PARENT_OF', 'Myrcella Baratheon'],
    ['Robert I Baratheon', 'PARENT_OF', 'Myrcella Baratheon'],
    ['Cersei Lannister', 'PARENT_OF', 'Myrcella Baratheon'],
    ['Cersei Lannister', 'PARENT_OF', 'Brandon Stark'],
    ["Tywin Lannister", 'PARENT_OF', 'Jaime Lannister'],
    ["Missandei", 'SPOUSE', 'Grey Worm'],
    ["Brienne of Tarth", 'SPOUSE', 'Jaime Lannister']
])

## we need to map the above triples to the id's which we used in our training/testing.
## This information is stored in the triple factory "got", which we created at the beginning

# unseen_filter = np.array(list({tuple(i) for i in np.vstack((positives_filter, X_unseen))}))
#     filter_triples=unseen_filter,   # Corruption strategy filter defined above
#     corrupt_side = 's+o',
#     use_default_protocol=False, # corrupt subj and obj separately while evaluating
#     verbose=True
# )


In [ ]:
from pykeen import predict
pack = predict.predict_triples(model=pipeline_result.model, triples=X_unseen, triples_factory=got)

In [ ]:
# scores are real numbers that need to be translated into probabilities [0,1] 
# for this, we use the expit transform.

from scipy.special import expit
processed_results = pack.process(factory=got).df

probs = expit(processed_results['score'])

processed_results['prob'] = probs
processed_results['triple'] = list(zip([' '.join(x) for x in X_unseen]))

# processed_results
pd.DataFrame(list(zip([' '.join(x) for x in X_unseen],  
                      np.squeeze(processed_results['score']),
                      np.squeeze(probs))), 
             columns=['statement', 'score', 'prob']).sort_values("score", ascending=False)

NB : the probabilities are not calibrated in any sense. To calibrate them, one may use a procedure such as [Platt scaling](https://en.wikipedia.org/wiki/Platt_scaling) or [Isotonic regression](https://en.wikipedia.org/wiki/Isotonic_regression). The challenge is to define what is a true triple and what is a false one, as the calibration of the probability of a triple being true depends on the base rate of positives and negatives.

### Exercise 5

Analyse the results in the tables. Some predicted links are very likely to be true, others  capture things that never really happened. Can you spot which ones?

# 6 Visualisation

[Tensorboard](https://www.tensorflow.org/tensorboard) allows to dig into the workings of our model, plot how it is learning, and visualize [high-dimensional embeddings](https://projector.tensorflow.org/). See [this tutorial](https://www.tensorflow.org/tensorboard/get_started) to get started with Tensorflow and see [here](https://pykeen.readthedocs.io/en/stable/tutorial/trackers/using_tensorboard.html) for Tensorboard with PyKEEN.

First ytou neeed to start the tensorboard web application from the command line with 

$ tensorboard --logdir=~/.data/pykeen/logs/tensorboard/

and then we can add tensorboard as the result_tracker in our pipeline.

In [ ]:
pipeline_result = pipeline(
    model='ComplEx',
    training=royals_training,
    testing=royals_testing,
    training_kwargs=dict(
        num_epochs=200
    ),
    dimensions=150,
    optimizer='adam',
    optimizer_kwargs={'lr':1e-3},
    loss='pairwisehinge', 
    regularizer='LP', 
    regularizer_kwargs={'p':3, 'weight':1e-5}, 
    negative_sampler='basic',
    negative_sampler_kwargs=dict(
        filtered=True,
    ),
    result_tracker='tensorboard'
)

### Exercise 7 Your Own Data now

Choose a dataset of your own. Best if it is the data you are using in your group project. 

- Create a training and testset. 
- Train your model to compute Knowledge Graph Embeddings, and save the best parameters model. - Predict new links over your dataset
- Visualise the embeddings you computed 
- Optional : cluster your embeddings, [see this tutorial](https://docs.ampligraph.org/en/1.4.0/tutorials/ClusteringAndClassificationWithEmbeddings.html)